In [1]:
"""
07_logistic_regression.py

Baseline model: logistic regression on the financial ratios.

Train/test split is by bookyear (2017-2018 train, 2019 test), not a
random row split - see utils/time_split.py for why. Winsorization is
fit on the training fold only, inside the pipeline (utils/winsorizer.py).
"""


'\n07_logistic_regression.py\n\nBaseline model: logistic regression on the financial ratios.\n\nTrain/test split is by bookyear (2017-2018 train, 2019 test), not a\nrandom row split - see utils/time_split.py for why. Winsorization is\nfit on the training fold only, inside the pipeline (utils/winsorizer.py).\n'

In [2]:
from utils.load_data_features import load_data_features

df = load_data_features()


In [3]:
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from config import FEATURES_FINANCIAL, RANDOM_STATE, TARGET, WINSOR_COLUMNS
from utils.model_results import save_model_results
from utils.print_section import print_section
from utils.time_split import time_based_split
from utils.winsorizer import Winsorizer

MODEL_NAME = "Logistic Regression"

X_train, X_test, y_train, y_test = time_based_split(df, FEATURES_FINANCIAL, TARGET)

pipeline = Pipeline([
    ("winsorizer", Winsorizer(columns=WINSOR_COLUMNS)),
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------------
# Performance
#
# Accuracy is not informative here (failure rate is <1%, so always
# predicting "healthy" already scores ~99%). PR-AUC is added next to
# ROC-AUC because it is more informative under strong class imbalance.
# ------------------------------------------------------------------
print_section("Performance")

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "pr_auc": average_precision_score(y_test, y_prob),
}

for name, value in metrics.items():
    print(f"{name:10s}: {value:.4f}")

print_section("Confusion matrix")
print(confusion_matrix(y_test, y_pred))

print_section("Classification report")
print(classification_report(y_test, y_pred, zero_division=0))

print_section("Coefficients")

coefs = pd.DataFrame({
    "feature": FEATURES_FINANCIAL,
    "coefficient": pipeline.named_steps["model"].coef_[0],
}).sort_values("coefficient")

print(coefs)

# ------------------------------------------------------------------
# Baseline comparison
# ------------------------------------------------------------------
print_section("Model improvement over baseline")

baseline_rate = y_train.mean()
print(f"Baseline failure rate (train): {baseline_rate:.4%}")
print(f"Model precision: {metrics['precision']:.4%}")
print(f"Improvement factor: {metrics['precision'] / baseline_rate:.2f}x")

save_model_results(MODEL_NAME, metrics)



Performance
accuracy  : 0.8108
precision : 0.0065
recall    : 0.6527
f1        : 0.0130
roc_auc   : 0.8024
pr_auc    : 0.0097

Confusion matrix
[[263184  61306]
 [   215    404]]

Classification report


              precision    recall  f1-score   support

           0       1.00      0.81      0.90    324490
           1       0.01      0.65      0.01       619

    accuracy                           0.81    325109
   macro avg       0.50      0.73      0.45    325109
weighted avg       1.00      0.81      0.89    325109


Coefficients
         feature  coefficient
0  profitability    -1.259065
4        log_age    -0.214406
2       solvency    -0.144501
5           size    -0.062808
1      liquidity    -0.039148
3      structure     1.426189

Model improvement over baseline
Baseline failure rate (train): 0.3355%
Model precision: 0.6547%
Improvement factor: 1.95x
